## Reporting Development

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

import os
import json
import typing
from copy import deepcopy
from pathlib import Path
from IPython.display import Image as IPyImage

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pprint import pprint

from fibsem.structures import FibsemImage
from fibsem.applications.autolamella.tools.data import calculate_statistics_dataframe
from fibsem.applications.autolamella.structures import (
    Experiment,
    AutoLamellaStage,
    MILL_POLISHING_KEY,
)
from fibsem.applications.autolamella.tools.reporting import (
    generate_final_overview_image,
)
from adaptive_polish.strategy.adaptive_polish import (
    AdaptivePolishMillingConfig,
    AdaptivePolishMillingStrategy,
)

if typing.TYPE_CHECKING:
    from os import PathLike


pd.set_option("display.max_columns", None)  # Show all columns in the DataFrame
pd.set_option("display.max_rows", None)  # Show all rows in the DataFrame


## Utils for Re-running Model Inference

In [ ]:
def get_adaptive_polishing_dataframe_from_inference(
    exp: Experiment,
    experiment_path: str | PathLike[str],
    strategy: AdaptivePolishMillingStrategy,
) -> pd.DataFrame:
    experiment_path = Path(experiment_path)
    dat = []
    for pos in exp.positions:
        print(f"{pos.info}")
        position_path = experiment_path / pos.name

        filenames = sorted(
            position_path.glob(os.path.join("adaptive*", "sem", "*.tif"))
        )
        for i, filename in enumerate(filenames):
            sem_image = FibsemImage.load(str(filename))

            stages = [
                stage
                for stage in pos.milling_workflows[MILL_POLISHING_KEY]
                if isinstance(stage.strategy, AdaptivePolishMillingStrategy)
            ]
            if stages:
                stage = stages[-1]
            else:
                print("Unable to get lamella info without stage")
                continue

            info = strategy._get_lamella_info(
                i,
                f"cycle-{i}",
                sem_image,
                None,  # type: ignore
                milling_stage=stage,
                lamella_pad_x=0,
            )
            ddict = {
                "experiment_name": exp.name,
                "experiment_id": exp._id,
                "lamella": pos.name,
                "lamella_id": pos._id,
                "milling_cycle": i,
                "milling_time_s": info.statistics.milling_time_s,
                "lamella_area_um2": info.statistics.lamella_area_um2,
                "min_gis_um": info.statistics.min_GIS_um,
                "crack_area_um2": info.statistics.crack_area_um2,
            }
            if (
                info.statistics.gis_thickness_filtered_um is not None
                and info.statistics.xlims_px is not None
            ):
                ddict["gis_thickness_mean_um"] = np.mean(
                    info.statistics.gis_thickness_filtered_um[
                        info.statistics.xlims_px[0] : info.statistics.xlims_px[1] + 1
                    ]
                )

            dat.append(ddict)

    df = pd.DataFrame(dat)
    return df

In [ ]:
def get_adaptive_polishing_dataframe(
    exp: Experiment, experiment_path: str | PathLike[str]
) -> pd.DataFrame:
    dat: list[dict[str, typing.Any]] = []
    experiment_path = Path(experiment_path)
    for pos in exp.positions:
        print(f"Position: {pos.name}")

        position_path = experiment_path / pos.name

        filenames = position_path.glob("adaptive*")
        if not filenames:
            print(f"No adaptive subdirectory found for position {pos.name}")
            continue

        fname = list(
            position_path.glob(os.path.join("adaptive*", "*GIS_thickness.json"))
        )
        if not fname:
            print(f"No GIS thickness JSON found for position {pos.name}")
            continue
        with open(fname[0], "r") as f:
            ddict = dict(json.load(f))

        detail_ddict = None
        detail_fname = list(
            position_path.glob(
                os.path.join("adaptive*", "*GIS_thickness_detailed.json")
            )
        )
        if not detail_fname:
            print(f"No Detailed GIS thickness JSON found for position {pos.name}")
            continue
        with open(detail_fname[0], "r") as f:
            detail_ddict = dict(json.load(f))
        for k, v in ddict["image"].items():
            mean_gis_thickness_um = 0
            if detail_ddict is not None and k in detail_ddict["image"]:
                mean_gis_thickness_um = np.mean(
                    detail_ddict["gis_thickness_filtered_um"][k]
                )

            dat.append(
                deepcopy(
                    {
                        "experiment_id": exp._id,
                        "experiment_name": exp.name,
                        "lamella": pos.name,
                        "lamella_id": pos._id,
                        "milling_cycle": k,
                        "identifer": v,
                        "milling_time_s": ddict["milling_time_s"][k],
                        "min_GIS_um": ddict["min_GIS_um"][k],
                        "mean_GIS_thickness_um": mean_gis_thickness_um,
                        "crack_area_um2": ddict["crack_area_um2"][k],
                        "lamella_area_um2": ddict.get("lamella_area_um2", {}).get(
                            k, None
                        ),
                    }
                )
            )

    df = pd.DataFrame(dat)
    if df.empty:
        print("No adaptive polishing data found.")
        return df
    df = df.sort_values(by=["lamella", "milling_cycle"])
    df = df.reset_index(drop=True)

    return df

In [ ]:
# Specify paths
experiment_path = Path()

model_path = Path()

In [ ]:
config = AdaptivePolishMillingConfig(model_path=str(model_path))
strategy = AdaptivePolishMillingStrategy(config=config)
strategy._load_model()

exp: Experiment = Experiment.load(experiment_path / "experiment.yaml")
filenames = list(experiment_path.glob("overview-*.tif"))
overview_image = filenames[-1] if filenames else None
if overview_image is not None:
    image = FibsemImage.load(str(overview_image))

    # overview image display
    fig = generate_final_overview_image(exp, image)
    plt.show()

# dataframes
dfs = calculate_statistics_dataframe(experiment_path)

df_history = dfs[1]
df_steps = dfs[3]
df_milling = dfs[-1]

# group df_steps by stage and step, then aggregate the mean of duration, and sum of duration
df_steps_grouped = (
    df_steps.groupby(["stage", "step"]).agg({"duration": "mean"}).reset_index()
)
# exclude stage: Created, Finished
# exclude step: STARTED, FINISHED, NULL_END
df_steps_grouped = df_steps_grouped[
    ~df_steps_grouped["stage"].isin(["Created", "Finished"])
]
df_steps_grouped = df_steps_grouped[
    ~df_steps_grouped["step"].isin(["STARTED", "FINISHED", "NULL_END"])
]

# format duration in MM:SS
df_steps_grouped["duration"] = df_steps_grouped["duration"].apply(
    lambda x: f"{int(x // 60):02}:{int(x % 60):02}"
)
display(df_steps_grouped)

# group by step and name, then aggregate the mean of duration
df_milling_grouped = (
    df_milling.groupby(["stage", "step", "name"])
    .agg({"duration": "mean"})
    .reset_index()
)
# format duration in MM:SS
df_milling_grouped["duration"] = df_milling_grouped["duration"].apply(
    lambda x: f"{int(x // 60):02}:{int(x % 60):02}"
)
display(df_milling_grouped)


# adaptive polishing dataframes
df_adaptive_polishing = get_adaptive_polishing_dataframe(exp, experiment_path)

# this is expensive, so don't run unless we want lamella area
df_adaptive_polishing_from_inference = get_adaptive_polishing_dataframe_from_inference(
    exp, experiment_path=experiment_path, strategy=strategy
)
# display the dataframes for each position


In [ ]:
dat = []
states = [
    AutoLamellaStage.SetupLamella,
    AutoLamellaStage.MillRough,
    AutoLamellaStage.MillPolishing,
]

for pos in exp.positions:
    ddict: dict[str, typing.Any] = {
        "experiment_id": exp._id,
        "experiment_name": exp.name,
        "lamella_id": pos._id,
        "lamella": pos.name,
    }

    for state in states:
        state_obj = pos.states.get(state)
        if state_obj:
            ddict["stage"] = state_obj.stage.name
            ddict["start_timestamp"] = state_obj.start_timestamp
            ddict["end_timestamp"] = state_obj.end_timestamp
            ddict["completed"] = state_obj.completed_at
            ddict["duration"] = state_obj.duration_str

        dat.append(deepcopy(ddict))

df_exp = pd.DataFrame(dat)
display(df_exp)

In [ ]:
# for df_steps, group by stage and step, then aggregate the mean of duration
df_steps_grouped = (
    df_steps.groupby(["stage", "step"]).agg({"duration": "mean"}).reset_index()
)
# exclude stage: Created, Finished
df_steps_grouped = df_steps_grouped[
    ~df_steps_grouped["stage"].isin(["Created", "Finished"])
]
# exclude step: STARTED, FINISHED, NULL_END
df_steps_grouped = df_steps_grouped[
    ~df_steps_grouped["step"].isin(["STARTED", "FINISHED", "NULL_END"])
]

# if step contains the string "MILL_*", group it as MILLING, otherwise OVERHEAD
df_steps_grouped["step"] = df_steps_grouped["step"].apply(
    lambda x: "MILLING" if "MILL_" in x else "OVERHEAD"
)  # NOTE: as fiducial milling is part of SETUP_PATTERNS its not included in MILLING
# group by stage and step, then aggregate the mean of duration
df_steps_grouped = (
    df_steps_grouped.groupby(["stage", "step"]).agg({"duration": "sum"}).reset_index()
)
# format duration in MM:SS
df_steps_grouped["duration"] = df_steps_grouped["duration"].apply(
    lambda x: f"{int(x // 60):02}:{int(x % 60):02}"
)
display(df_steps_grouped)


In [ ]:
for pos in exp.positions:
    position_path = experiment_path / pos.name

    filenames = position_path.glob("adaptive*")
    if not filenames:
        print(f"No adaptive subdirectory found for position {pos.name}")
        continue

    fname = list(position_path.glob(os.path.join("adaptive*", "*centring.png")))
    if fname:
        display(IPyImage(filename=fname[0]))

    fname = list(position_path.glob(os.path.join("adaptive*", "*GIS_thickness.png")))
    if fname:
        image = display(IPyImage(filename=fname[0]))

    fname = list(position_path.glob(os.path.join("adaptive*", "*GIS_thickness.json")))
    if not fname:
        print(f"No GIS thickness JSON found for position {pos.name}")
        continue
    with open(fname[0], "r") as f:
        dat = json.load(f)

    pprint(dat)


In [ ]:
for pos in exp.positions:
    print(f"Position: {pos.name}")
    position_path = experiment_path / pos.name

    # display all the images in the sem and fib directories as subplots
    sem_images = sorted(position_path.glob(os.path.join("adaptive*", "sem", "*.tif")))
    fib_images = sorted(position_path.glob(os.path.join("adaptive*", "fib", "*.tif")))

    if sem_images and fib_images:
        fig, axs = plt.subplots(len(sem_images), 2, figsize=(7, len(sem_images) * 2))
        axs = axs.flat
        images = []
        for i, (sem_path, fib_path) in enumerate(zip(sem_images, fib_images)):
            sem_img = Image.open(sem_path)
            fib_img = Image.open(fib_path)

            sem_idx = i * 2
            fib_idx = sem_idx + 1
            axs[sem_idx].imshow(sem_img, cmap="gray")
            axs[sem_idx].set_title(f"SEM Image {i + 1}")
            axs[sem_idx].axis("off")
            axs[fib_idx].imshow(fib_img, cmap="gray")
            axs[fib_idx].set_title(f"FIB Image {i + 1}")
            axs[fib_idx].axis("off")

            combined_img = Image.new(
                "RGB", (sem_img.width + fib_img.width, sem_img.height)
            )

            # Collate for GIF
            combined_img.paste(sem_img, (0, 0))
            combined_img.paste(fib_img, (sem_img.width, 0))
            images.append(combined_img)

        fig.tight_layout()
        fig.show()

        if images:
            # compress the images to reduce file size
            images = [
                img.convert("P", palette=Image.Palette.ADAPTIVE, colors=256)
                for img in images
            ]
            gif_path = position_path / "adaptive.gif"
            # save as gif
            images[0].save(
                gif_path,
                save_all=True,
                append_images=images[1:],
                duration=250,
                loop=0,
            )
            display(IPyImage(filename=gif_path))
            print(f"GIF saved for position {pos.name} at {gif_path}")


In [ ]:
# plot a line plot for lamella_area_um2, gis_thickness_mean_um, min_gis_um, crack_area_um2
import seaborn as sns

df = df_adaptive_polishing_from_inference
sns.set(style="whitegrid")
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df, x="milling_cycle", y="lamella_area_um2", hue="lamella", marker="o"
)
plt.title("Lamella Area over Milling Cycles")
plt.xlabel("Milling Cycle")
plt.ylabel("Lamella Area (µm²)")
plt.legend(title="Lamella")
plt.show()

# plot a line plot for gis_thickness_mean_um and min_gis_um
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df, x="milling_cycle", y="gis_thickness_mean_um", hue="lamella", marker="o"
)
plt.title("GIS Thickness Mean and Minimum over Milling Cycles")
plt.xlabel("Milling Cycle")

plt.ylabel("GIS Thickness Mean (µm)")
plt.legend(title="Lamella")
plt.show()


# plot a line plot for min_gis_um
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x="milling_cycle", y="min_gis_um", hue="lamella", marker="o")
plt.title("Minimum GIS Thickness over Milling Cycles")
plt.xlabel("Milling Cycle")
plt.ylabel("Minimum GIS Thickness (µm)")
plt.legend(title="Lamella")
plt.show()

# plot a line plot for crack_area_um2
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x="milling_cycle", y="crack_area_um2", hue="lamella", marker="o")
plt.title("Crack Area over Milling Cycles")
plt.xlabel("Milling Cycle")
plt.ylabel("Crack Area (µm²)")
plt.legend(title="Lamella")
plt.show()

In [ ]:
# Casper: I added this block below, as we do not have seaborn installed in this enviroment
import matplotlib.pyplot as plt

df = df_adaptive_polishing_from_inference


# Helper function to plot a line plot with multiple "lamella" lines
def plot_line(df, y_col, title, y_label) -> None:
    plt.figure(figsize=(12, 6))
    for lamella_id, group in df.groupby("lamella"):
        plt.plot(
            group["milling_cycle"],
            group[y_col],
            marker="o",
            label=f"Lamella {lamella_id}",
        )
    plt.title(title)
    plt.xlabel("Milling Cycle")
    plt.ylabel(y_label)
    plt.legend(title="Lamella")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Lamella Area over Milling Cycles
plot_line(
    df, "lamella_area_um2", "Lamella Area over Milling Cycles", "Lamella Area (µm²)"
)

# GIS Thickness Mean over Milling Cycles
plot_line(
    df,
    "gis_thickness_mean_um",
    "GIS Thickness Mean over Milling Cycles",
    "GIS Thickness Mean (µm)",
)

# Minimum GIS Thickness over Milling Cycles
plot_line(
    df,
    "min_gis_um",
    "Minimum GIS Thickness over Milling Cycles",
    "Minimum GIS Thickness (µm)",
)

# Crack Area over Milling Cycles
plot_line(df, "crack_area_um2", "Crack Area over Milling Cycles", "Crack Area (µm²)")


### Export StrategyRunInformation as DataFrame

In [9]:
%load_ext autoreload
%autoreload 2
from adaptive_polish._dataclasses import (
    StrategyRunInformation,
    LamellaStatistics,
    StopReasons,
    CycleInformation,
    StrategyTimestamps,
    CycleTimestamps,
    ProcessTimestamps,
)

import time
import pandas as pd
pd.set_option("display.max_columns", None)  # Show all columns in the DataFrame

cycle_info: list[CycleInformation] = []
cycle_start = time.time()

for i in range(5):
    cycle_base = cycle_start + i * 45  # start each cycle 45 s apart
    timestamps = CycleTimestamps(
        cycle=ProcessTimestamps(start=cycle_base, end=cycle_base + 40),
        image=ProcessTimestamps(start=cycle_base + 1, end=cycle_base + 6),
        process=ProcessTimestamps(start=cycle_base + 6, end=cycle_base + 15),
        update_stage=ProcessTimestamps(start=cycle_base + 15, end=cycle_base + 22),
        predict=ProcessTimestamps(start=cycle_base + 22, end=cycle_base + 26),
        check_lamella=ProcessTimestamps(start=cycle_base + 26, end=cycle_base + 31),
        mill=ProcessTimestamps(start=cycle_base + 31, end=cycle_base + 40),
    )

    cycle_info.append(
        CycleInformation(
            milling_cycle=i,
            identifier=f"cycle-{i}",
            timestamps=timestamps,
            lamella_statistics=LamellaStatistics(
                image_pixel_size_m=(0.1, 0.1),
                prediction_pixel_size_m=(100e-9, 100e-9),
                crack_count=0,
                lamella_thickness_prediction_px=[],
                crack_thickness_prediction_px=[],
            ),
        )
    )
run_info = StrategyRunInformation(
    strategy_name="test-strategy",
    stage_name="polishing-stage",
    lamella_name="01-fun-dog",
    timestamps=StrategyTimestamps(
        strategy=ProcessTimestamps(
            start=time.time() - 360,
            end=time.time(),
        ),
        setup=ProcessTimestamps(
            start=time.time() - 350,
            end=time.time() - 180,
        ),
        centre_lamella=ProcessTimestamps(
            start=time.time() - 350,
            end=time.time() - 340,
        )
    ),
    strategy_end_reason=StopReasons.MIN_GIS_THICKNESS.name,
    cycle_information=cycle_info
)

print(run_info)

# df = run_info.to_dataframe()
# display(df)


df = run_info.to_summary_dataframe()
display(df)

df = run_info.to_final_dataframe()
display(df)




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
StrategyRunInformation(strategy_name='test-strategy', stage_name='polishing-stage', lamella_name='01-fun-dog', timestamps=StrategyTimestamps(strategy=ProcessTimestamps(start=1759195452.205242, end=1759195812.2052422, exception=None), setup=ProcessTimestamps(start=1759195462.2052426, end=1759195632.2052429, exception=None), centre_lamella=ProcessTimestamps(start=1759195462.205243, end=1759195472.205243, exception=None)), strategy_end_reason='MIN_GIS_THICKNESS', cycle_information=[CycleInformation(milling_cycle=0, identifier='cycle-0', timestamps=CycleTimestamps(cycle=ProcessTimestamps(start=1759195812.2050366, end=1759195852.2050366, exception=None), image=ProcessTimestamps(start=1759195813.2050366, end=1759195818.2050366, exception=None), process=ProcessTimestamps(start=1759195818.2050366, end=1759195827.2050366, exception=None), update_stage=ProcessTimestamps(start=1759195827.2050366, end=175919583

,Strategy Name,Stage Name,Lamella Name,Strategy End Reason,Strategy Strategy Duration,Strategy Setup Duration,Strategy Centre Lamella Duration,Cycle Cycle Duration,Cycle Image Duration,Cycle Process Duration,Cycle Update Stage Duration,Cycle Predict Duration,Cycle Check Lamella Duration,Cycle Mill Duration,Lamella Area (um2),GIS Thickness Min (um),GIS Thickness Mean (um),Crack Count
0,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
1,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
2,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
3,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
4,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0


,Strategy Name,Stage Name,Lamella Name,Strategy End Reason,Strategy Strategy Duration,Strategy Setup Duration,Strategy Centre Lamella Duration,Cycle Cycle Duration,Cycle Image Duration,Cycle Process Duration,Cycle Update Stage Duration,Cycle Predict Duration,Cycle Check Lamella Duration,Cycle Mill Duration,Lamella Area (um2),GIS Thickness Min (um),GIS Thickness Mean (um),Crack Count
0,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0


In [ ]:
df = run_info.to_full_dataframe()
display(df)


,milling_cycle,identifier,timestamps.cycle.start,timestamps.cycle.end,timestamps.cycle.exception,timestamps.image.start,timestamps.image.end,timestamps.image.exception,timestamps.process.start,timestamps.process.end,timestamps.process.exception,timestamps.update_stage.start,timestamps.update_stage.end,timestamps.update_stage.exception,timestamps.predict.start,timestamps.predict.end,timestamps.predict.exception,timestamps.check_lamella.start,timestamps.check_lamella.end,timestamps.check_lamella.exception,timestamps.mill.start,timestamps.mill.end,timestamps.mill.exception,lamella_statistics.image_pixel_size_m,lamella_statistics.prediction_pixel_size_m,lamella_statistics.crack_count,lamella_statistics.lamella_thickness_prediction_px,lamella_statistics.crack_thickness_prediction_px,lamella_statistics.estimated_milling_time_s,lamella_statistics.lamella_bounding_box_prediction_px,lamella_statistics.xlims_prediction_px,lamella_statistics.lamella_bounding_box_image_px,lamella_statistics.xlims_image_px,lamella_statistics.gis_thickness_image_px,lamella_statistics.gis_thickness_filtered_image_px,lamella_statistics.gis_thickness_min_image_px,lamella_statistics.gis_thickness_median_image_px,lamella_statistics.gis_thickness_mean_image_px,strategy_name,stage_name,lamella_name,strategy_end_reason,timestamps.strategy.start,timestamps.strategy.end,timestamps.strategy.exception,timestamps.setup.start,timestamps.setup.end,timestamps.setup.exception,timestamps.centre_lamella.start,timestamps.centre_lamella.end,timestamps.centre_lamella.exception,timestamps.cycle_duration,timestamps.image_duration,timestamps.process_duration,timestamps.update_stage_duration,timestamps.predict_duration,timestamps.check_lamella_duration,timestamps.mill_duration,timestamps.strategy_duration,timestamps.setup_duration,timestamps.centre_lamella_duration
0,0,cycle-0,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759195e+09,None,40.0,5.0,9.0,7.0,4.0,5.0,9.0,360.0,170.0,10.0
1,1,cycle-1,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759195e+09,None,40.0,5.0,9.0,7.0,4.0,5.0,9.0,360.0,170.0,10.0
2,2,cycle-2,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759195e+09,None,40.0,5.0,9.0,7.0,4.0,5.0,9.0,360.0,170.0,10.0
3,3,cycle-3,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759196e+09,None,1.759195e+09,1.759195e+09,None,40.0,5.0,9.0,7.0,4.0,5.0,9.0,360.0,170.0,10.0
4,4,cycle-4,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09

In [11]:
df = run_info.to_summary_dataframe()
display(df)

,Strategy Name,Stage Name,Lamella Name,Strategy End Reason,Strategy Strategy Duration,Strategy Setup Duration,Strategy Centre Lamella Duration,Cycle Cycle Duration,Cycle Image Duration,Cycle Process Duration,Cycle Update Stage Duration,Cycle Predict Duration,Cycle Check Lamella Duration,Cycle Mill Duration,Lamella Area (um2),GIS Thickness Min (um),GIS Thickness Mean (um),Crack Count
0,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
1,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
2,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
3,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0
4,test-strategy,polishing-stage,01-fun-dog,MIN_GIS_THICKNESS,360.0,170.0,10.0,40.0,5.0,9.0,7.0,4.0,5.0,9.0,0.0,None,None,0


In [15]:
df = run_info.to_dataframe2()

df2 = pd.json_normalize(df["cycle_information"][0])

display(df2)

,milling_cycle,identifier,timestamps.cycle.start,timestamps.cycle.end,timestamps.cycle.exception,timestamps.image.start,timestamps.image.end,timestamps.image.exception,timestamps.process.start,timestamps.process.end,timestamps.process.exception,timestamps.update_stage.start,timestamps.update_stage.end,timestamps.update_stage.exception,timestamps.predict.start,timestamps.predict.end,timestamps.predict.exception,timestamps.check_lamella.start,timestamps.check_lamella.end,timestamps.check_lamella.exception,timestamps.mill.start,timestamps.mill.end,timestamps.mill.exception,lamella_statistics.image_pixel_size_m,lamella_statistics.prediction_pixel_size_m,lamella_statistics.crack_count,lamella_statistics.lamella_thickness_prediction_px,lamella_statistics.crack_thickness_prediction_px,lamella_statistics.estimated_milling_time_s,lamella_statistics.lamella_bounding_box_prediction_px,lamella_statistics.xlims_prediction_px,lamella_statistics.lamella_bounding_box_image_px,lamella_statistics.xlims_image_px,lamella_statistics.gis_thickness_image_px,lamella_statistics.gis_thickness_filtered_image_px,lamella_statistics.gis_thickness_min_image_px,lamella_statistics.gis_thickness_median_image_px,lamella_statistics.gis_thickness_mean_image_px
0,0,cycle-0,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None
1,1,cycle-1,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None
2,2,cycle-2,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None
3,3,cycle-3,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None
4,4,cycle-4,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,1.759196e+09,1.759196e+09,None,"(0.1, 0.1)","(1e-07, 1e-07)",0,[],[],None,None,None,None,None,None,None,None,None,None
